# Multi-accent TTS frontend — full training run

Overnight run of the full pipeline on the complete Unilex lexicon (~116k words).

**Code comes in over git; results go out to Drive.** Both happen from inside the
Colab VM, so your employer's network filter is never in the path. The only thing
that touches your own machine is the Drive auth popup in section 1.

## One-time setup before you start

**1. Push the pipeline to a PRIVATE GitHub repo.** Use the staged folder
(`github-repo/`) prepared for you — it already has the right layout.

> ⚠️ **Private, not public.** Unilex is a licensed CSTR/Edinburgh lexicon and is
> not freely redistributable. Keep the repo private.

**2. Give Colab a token.** Click the 🔑 **Secrets** icon in the left sidebar →
*Add new secret* → name it `GH_TOKEN`, paste a GitHub personal access token with
`repo` scope, and enable *Notebook access*. Never paste the token into a cell.

**3. Make a `frontend` folder** at the top level of your Google Drive (My Drive).
Nothing needs uploading into it — checkpoints and the prepared dataset get
written there by the notebook.

**4.** Runtime → Change runtime type → **L4 GPU**, then run the cells in order.

**Do not use Runtime → Run all.** Section 6 is the disconnect-recovery path and
would start a second training run on top of the first.

**Why results go to Drive.** Colab disconnects — Pro caps sessions around 12h and
drops idle tabs sooner. Checkpoints (115 MB each) land on Drive every 1000 steps,
so a disconnect costs minutes, never the run. Section 6 resumes.


## 0. Check the GPU

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

If `CUDA` is False, stop: Runtime → Change runtime type → GPU. This model does
not train usefully on CPU.

## 1. Mount Drive and set the paths


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# --- where the CODE comes from (git, cloned inside the VM) ----------------
GIT_REPO     = 'calloumi123/multi-accent-tts-frontend'
GIT_BRANCH   = 'main'
TOKEN_SECRET = 'GH_TOKEN'              # name of the Colab secret holding your PAT
CODE         = Path('/content/frontend')

# --- where RESULTS go (Drive) --------------------------------------------
DRIVE   = Path('/content/drive/MyDrive/frontend')    # outputs only, no uploads needed

# --- accents ---------------------------------------------------------------
# The THREE REALISED accent lexicons, which ship in the research repo at
# notebooks/lex/. These are what make this a multi-accent model:
#   edi = Edinburgh Scottish (rhotic, tapped r)   'run' -> t^ uh n
#   gam = General American   (rhotic)             'bath' -> b a th
#   rpx = RP                 (NON-rhotic)         'bath' -> b aa th, 'for' -> f @
# All three have the identical 116,739 headwords as the master file, so using
# them costs ZERO coverage -- no extra lines are dropped by the all-accents-
# must-resolve rule. Resolved in section 3, after the clone.
ACCENTS    = ['edi', 'gam', 'rpx']
ACCENT_LEX = {}     # {accent: Path to unilex-<accent>.out}

# The MASTER lexicon is still needed, but only for coverage_corpus.py -- it is
# the only file carrying the unigram frequencies. It is not used for targets.
LEX_IN_DRIVE = DRIVE / 'unilex'
LEX          = None

RUNS       = DRIVE / 'runs'             # checkpoints -> Drive, survive a disconnect
DS_ARCHIVE = DRIVE / 'dataset.tar.gz'   # prepared data, so you never re-prep
DATASET    = Path('/content/dataset')   # working copy on fast local disk
WORK       = Path('/content/work')

# --- corpus sources -------------------------------------------------------
# 'take'   = raw sentences pulled from that source
# 'yield_' = fraction that survives cleaning (digit filter + OOV), MEASURED on
#            your actual unilex lexicon using 20k-line samples per source.
# usable ~= take * yield_
# Sized for THREE accents: every sentence becomes 3 training items (one src,
# three different tgt), so steps/epoch triples. ~1M usable sentences x 3 accents
# gives the same overnight cost as 2.95M sentences of a single-accent model --
# and news text saturates at ~55-70% lexicon coverage anyway (measured), with
# the coverage corpus supplying the rest regardless of volume.
SOURCES = {
    'books':    dict(take=700_000, yield_=0.71),  # Gutenberg prose (LibriSpeech book corpus)
    'wmt_news': dict(take=400_000, yield_=0.40),  # WMT news-crawl 2020 (streams, stops early)
    'leipzig':  dict(take=500_000, yield_=0.46),  # Leipzig news 2020 (max 1M available)
}
ADD_COVERAGE = True    # + ~116k lines, one per lexicon headword -> drives OOV to ~0

# --- training -------------------------------------------------------------
EPOCHS    = 300    # a cap, not a target -- see section 5
BATCH     = 128    # was 32. The model is only 10M params (~160MB of weights+grads+
                   # Adam state), so a 24GB L4 is nowhere near VRAM-bound. Bigger
                   # batches are the cheapest speedup available: ~4x the samples
                   # per step for ~1.5-2x the step time.
BATCH_GROUP = 512  # MUST scale with BATCH. The loader length-sorts every item, then
                   # shuffles inside windows of this size. If it is <= BATCH you get
                   # no effective shuffling at all. 4x BATCH keeps the repo's
                   # original ~3-batches-per-window behaviour.
LR        = 1e-4   # was 5e-5. sqrt-scaled for the 4x batch. Do NOT go higher
                   # blindly: with GMM attention, too high an LR stops the
                   # alignment ever going monotonic-diagonal. See section 5.
SAVE_STEP = 1000   # low on purpose, see section 4d
MAX_WORDS = 100    # the training loader drops src over 600 chars; this ~matches,
                   # so we don't spend prep time on lines training will discard

RUNS.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)

raw   = sum(s['take'] for s in SOURCES.values())
est   = sum(s['take'] * s['yield_'] for s in SOURCES.values()) + (116_000 if ADD_COVERAGE else 0)
items = est * len(ACCENTS)
print(f'accents               : {", ".join(ACCENTS)}')
print(f'raw sentences to pull : {raw:,}')
print(f'estimated usable      : ~{est:,.0f}')
print(f'training items        : ~{items:,.0f}   (usable x {len(ACCENTS)} accents)')
print(f'steps per epoch       : ~{items / BATCH:,.0f}   @ batch {BATCH}')

### Check Drive is mounted and writable


In [ ]:
# We only WRITE to Drive, so the check is that it is mounted and writable --
# there are no uploads to verify any more.
DRIVE.mkdir(parents=True, exist_ok=True)
probe = DRIVE / '.write_test'
probe.write_text('ok'); probe.unlink()
print(f'OK       Drive is mounted and writable: {DRIVE}')

if LEX_IN_DRIVE.is_file():
    print(f'OK       lexicon found on Drive ({LEX_IN_DRIVE.stat().st_size/1e6:.1f} MB)')
else:
    print(f'note     no lexicon on Drive -- will look for it in the git repo instead')


## 2. Install dependencies

In [ ]:
# Colab already ships a CUDA-enabled torch, so we use that.
#
# Do NOT `pip install -r requirements.txt` from the research repo: it is a conda
# explicit-spec list pinned to python 3.7 / torch 1.9 / cuda 11.0 and will not
# resolve on Colab. The four packages below are all that train.py actually needs
# on top of torch.
!pip -q install nltk tqdm tensorboard scikit-learn matplotlib
# scikit-learn and matplotlib are NOT optional: train.py imports visual.py,
# which imports sklearn.decomposition.PCA at module load. Without them
# training dies on the very first import.

import nltk
# nltk renamed the tagger data in newer versions; request both names.
for pkg in ('averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng',
            'punkt', 'punkt_tab'):
    nltk.download(pkg, quiet=True)
print(nltk.pos_tag(['the', 'record', 'is', 'clear']), '<- tagger works')

## 3. Set up the code

Clones the research repo (which has `train.py`) and drops your data scripts in
beside it — they import each other by bare module name, so they must sit together.

In [ ]:
%cd /content
!rm -rf research frontend

# 1) the research training code -- public repo (train.py, datasets.py, layers, ...)
!git clone -q --depth 1 https://github.com/sunsiqitos/multi_accent_s2s_frontend research

# 2) YOUR private repo with the pipeline scripts (+ maybe the lexicon)
from google.colab import userdata
try:
    _tok = userdata.get(TOKEN_SECRET)
except Exception as e:
    _tok = None
    print(f'no Colab secret {TOKEN_SECRET!r} ({type(e).__name__}) -- trying unauthenticated')

_auth = f'{_tok}:x-oauth-basic@' if _tok else ''
!git clone -q --depth 1 --branch {GIT_BRANCH} https://{_auth}github.com/{GIT_REPO}.git frontend
# Scrub the token out of the clone so it is not left sitting in .git/config.
!cd frontend && git remote set-url origin https://github.com/{GIT_REPO}.git
del _tok, _auth

assert CODE.is_dir(), (
    f'clone of {GIT_REPO} failed. Check: repo name, branch {GIT_BRANCH!r}, and that '
    f'the {TOKEN_SECRET!r} secret exists with repo scope AND notebook access enabled.')

!cp /content/frontend/scripts/*.py /content/frontend/scripts/*.sh /content/research/
%cd /content/research

# The repo was written against sklearn ~0.24, where TSNE took `n_iter`.
# Current sklearn renamed it to `max_iter`, so the first eval step would
# raise TypeError -- and because no checkpoint exists yet at that point,
# train.py's cleanup handler DELETES the whole run folder. Fix it up front.
!sed -i 's/n_iter=300/max_iter=300/' visual.py
!grep -n 'max_iter' visual.py

# --- resolve the lexicon: Drive first, then the repo ----------------------
_cands = [LEX_IN_DRIVE, CODE / 'unilex', CODE / 'lex' / 'unilex']
LEX = next((c for c in _cands if c.is_file()), None)
assert LEX is not None, (
    'lexicon not found. Either upload `unilex` to ' + str(DRIVE) +
    ' or commit it to your repo as `unilex`. Looked in: ' +
    ', '.join(str(c) for c in _cands))
print(f'\nlexicon  -> {LEX}  ({LEX.stat().st_size/1e6:.1f} MB)')

# --- the three realised accent lexicons (they ship in the research repo) ---
ACCENT_LEX = {a: Path('/content/research/notebooks/lex') / f'unilex-{a}.out'
              for a in ACCENTS}
_missing = [str(p) for p in ACCENT_LEX.values() if not p.is_file()]
assert not _missing, f'accent lexicons missing from the research clone: {_missing}'
for a, p in ACCENT_LEX.items():
    print(f'accent   -> {a}: {p.name}  ({p.stat().st_size/1e6:.1f} MB)')

!ls -1 train.py prepare_data.py coverage_corpus.py unilex_master.py export_bundle.py


### Quick lexicon sanity check

`read` should come back with two different pronunciations (`r ii d` / `r e d`) and
`the` with a full and a reduced form. If those look right, the master file parsed
correctly.

In [ ]:
%cd /content/research
!python unilex_master.py "{LEX}"

## 4. Build the corpus and prepare the data

**Skip to section 4c if you already have `dataset.tar.gz` on Drive** from an
earlier session — no need to redo this.

### 4a. Corpus sources

Measured yields, run against your actual `unilex` on 20k-line samples per source:

| Source | Yield | Why |
|---|---|---|
| Gutenberg books | **70.9%** | Punctuated, mixed case, few digits. Best content match for read speech. |
| Leipzig news 2020 | **45.7%** | Digit-heavy; proper nouns not in the lexicon. |
| WMT news-crawl 2020 | **39.6%** | As above, plus longer sentences. |
| Coverage corpus | ~100% | Generated from the lexicon itself. |

So the mix is deliberately **books-heavy** — it yields nearly twice as much per
sentence pulled *and* better matches what a TTS frontend reads.

Two traps handled here, both found by testing:

1. **Leipzig is sorted alphabetically.** Slicing the head gave **0.1% kept** (all
   sentences starting `!`, `$`, `A…`). It is downloaded in full and shuffled.
   WMT news-crawl is already shuffled *and* deduped upstream, so that one can be
   streamed and cut short — no need to pull all 2.7 GB.
2. **Curly apostrophes `’` outnumber ASCII `'` 2:1** in news text. The tokeniser is
   `[A-Za-z']+`, so `wouldn’t` splits into `wouldn` + `t`, both OOV, dropping the
   whole line. All sources are normalised first.

Lines are also **deduplicated across sources** by hash, since these corpora overlap.

In [ ]:
%cd /content/research
import gzip, hashlib, random, tarfile, urllib.request
import fetch_corpus   # reuse the repo's own tested splitter / leipzig id-stripper

BOOKS_URL   = 'https://www.openslr.org/resources/11/librispeech-lm-corpus.tgz'
WMT_URL     = 'https://data.statmt.org/news-crawl/en/news.2020.en.shuffled.deduped.gz'
LEIPZIG_URL = 'https://downloads.wortschatz-leipzig.de/corpora/eng_news_2020_1M.tar.gz'

TRANS = str.maketrans({'’': "'", '‘': "'", '′': "'", '“': '"', '”': '"'})


def _download(url, dest):
    if dest.is_file() and dest.stat().st_size > 0:
        print(f'  cached: {dest.name}')
    else:
        print(f'  downloading {url}')
        urllib.request.urlretrieve(url, dest)
    return dest


def gen_books(take):
    # Gutenberg prose: punctuated and mixed-case, so it yields real _B breaks.
    outdir = WORK / 'books'
    # Check for the extracted tree FIRST -- otherwise a re-run after a crash
    # re-downloads 1.8 GB it does not need.
    if not outdir.is_dir():
        tgz = _download(BOOKS_URL, WORK / 'books.tgz')
        print('  extracting ~14k book files (a few minutes)')
        with tarfile.open(tgz) as t:
            t.extractall(outdir)
    files = sorted(outdir.rglob('corpus/*/*.txt'))
    random.Random(1234).shuffle(files)   # mix authors/eras instead of one shelf
    n = 0
    for fp in files:
        for s in fetch_corpus._iter_plain(fp, split_sentences=True):
            yield s
            n += 1
            if n >= take:
                return


def gen_wmt(take):
    # Shuffled and deduped upstream, so streaming the head is representative --
    # we stop early rather than downloading all 2.7 GB.
    with urllib.request.urlopen(WMT_URL) as resp:
        with gzip.open(resp, 'rt', encoding='utf-8', errors='replace') as f:
            for i, line in enumerate(f):
                if i >= take:
                    return
                yield line


def gen_leipzig(take):
    # Sorted alphabetically, so it MUST be shuffled before slicing.
    tgz = _download(LEIPZIG_URL, WORK / 'leipzig.tar.gz')
    lines = list(fetch_corpus._iter_leipzig(tgz))   # strips the leading "<id>\t"
    random.Random(1234).shuffle(lines)
    for s in lines[:take]:
        yield s


BUILDERS = {'books': gen_books, 'wmt_news': gen_wmt, 'leipzig': gen_leipzig}
print('builders ready:', ', '.join(BUILDERS))

### 4b. Build the corpus

Downloads, normalises and deduplicates. The books tarball is 1.8 GB and Leipzig
0.27 GB; both are cached in `/content/work`, so re-running after a crash does not
re-download. WMT is streamed and never fully downloaded.

In [ ]:
corpus = WORK / 'corpus.txt'
cov    = WORK / 'coverage.txt'

# Coverage corpus first (generated from the lexicon, so it needs no normalisation).
if ADD_COVERAGE and not cov.is_file():
    !python coverage_corpus.py --lexicon "{LEX}" --out "{cov}" \
        --mode sample --fillers 6 --sort freq

seen   = set()   # 8-byte hashes, for cross-source dedup
counts = {}

with corpus.open('w', encoding='utf-8') as out:
    for name, spec in SOURCES.items():
        print(f"\n== {name}  (take {spec['take']:,}) ==")
        n = dup = 0
        for line in BUILDERS[name](spec['take']):
            s = ' '.join(line.translate(TRANS).split())
            if len(s) < 8:
                continue
            h = hashlib.blake2b(s.encode('utf-8'), digest_size=8).digest()
            if h in seen:
                dup += 1
                continue
            seen.add(h)
            out.write(s + '\n')
            n += 1
        counts[name] = n
        print(f'  {n:,} unique written ({dup:,} duplicates skipped)')

    if ADD_COVERAGE:
        n = 0
        with cov.open(encoding='utf-8') as f:
            for line in f:
                out.write(line)
                n += 1
        counts['coverage'] = n
        print(f'\n  coverage corpus: {n:,} lines')

del seen
print(f"\ntotal raw corpus lines: {sum(counts.values()):,}")

### 4c. Shuffle

So training doesn't see all the books first and then all the news.

In [ ]:
!shuf "{corpus}" -o "{corpus}"
print('shuffled:', sum(1 for _ in corpus.open(encoding='utf-8')), 'lines')

### 4d. Run prepare_data

Measured at ~2,800 lines/sec, so a ~5M-line corpus takes roughly **30 minutes**.
It streams line by line, so memory stays flat regardless of corpus size.

In [ ]:
%cd /content/research

# One --lexicon per accent. prepare_data looks every word up in EVERY accent and
# drops the line if it is OOV in any of them, so the three accents stay exactly
# parallel (same src lines, three aligned tgt files). Because all three lexicons
# share the identical 116,739 headwords, nothing extra is dropped.
_lex_args = ' '.join(f'{a}:"{p}"' for a, p in ACCENT_LEX.items())

!python prepare_data.py \
    --input "{corpus}" \
    --lexicon {_lex_args} \
    --output-dir "{DATASET}" \
    --corpus-name full \
    --selector pos \
    --max-words {MAX_WORDS} \
    --val-frac 0.02 --test-frac 0.02 --seed 1234 \
    --report "{DATASET}/coverage.json"


**Read this report before training.** `coverage_rate` is the fraction of input
lines kept. A big `needs_normalisation_digits` means your text has lots of numbers;
`top_oov` lists the words costing you sentences. With the coverage corpus appended,
OOV should be near zero.

In [ ]:
import json
r = json.load(open(DATASET / 'coverage.json'))
print(json.dumps({k: v for k, v in r.items() if k != 'top_oov'}, indent=2))
print('\ntop OOV:', ', '.join(f'{w}({c})' for w, c in r['top_oov'][:15]) or 'none')

### 4e. Point the config at Colab, then archive the dataset to Drive

`/content` is wiped when the VM dies, so the prepared data is archived to Drive.
The config holds absolute `/content/dataset/...` paths, so on resume we restore to
that same location and the config stays valid.

In [ ]:
import json, shutil, subprocess

cfg_path = DATASET / 'train_config.json'
c = json.loads(cfg_path.read_text())
c['epochs']     = EPOCHS
c['batch_size'] = BATCH
# save_step defaults to 10000. Lowered on purpose: train.py calls
# remove_experiment_folder() on interrupt, which DELETES the run folder if it does
# not yet contain a checkpoint. On Colab interrupts happen, so we make sure a
# checkpoint lands early.
c['save_step']  = SAVE_STEP
c['batch_group_size'] = BATCH_GROUP
c['learning_rate']    = LR
cfg_path.write_text(json.dumps(c, indent=2))

print(json.dumps({k: c[k] for k in
      ('experiment_name','epochs','batch_size','batch_group_size','save_step',
       'r','attn_type','learning_rate')}, indent=2))

# The loader builds items per accent, so total items = sentences x accents.
_accents = list(c['data'].keys())
n_sent = sum(1 for _ in open(
    c['data'][_accents[0]]['corpus_1']['path_src'], encoding='utf-8'))
n_items = n_sent * len(_accents)
print(f'\naccents in config : {", ".join(_accents)}')
print(f'train sentences   : {n_sent:,}')
print(f'training items    : {n_items:,}   ({n_sent:,} x {len(_accents)} accents)')
print(f'steps per epoch   : {-(-n_items // BATCH):,}   @ batch {BATCH}')

# Archive as a single tarball -- far faster on Drive than many small files.
subprocess.run(['tar','czf',str(DS_ARCHIVE),'-C','/content','dataset'], check=True)
print('\narchived dataset ->', DS_ARCHIVE, f'({DS_ARCHIVE.stat().st_size/1e6:.1f} MB)')

## 5. Train

Launch TensorBoard first, then start training in the cell below it.

**Watch the attention alignment plot.** A clean monotonic diagonal means the model
is learning. If attention never aligns, lower `learning_rate` or the reduction
factor `r` and restart.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{RUNS}"

### Start training

Long-running — this is the overnight cell. Keep the browser tab open; free-tier
Colab drops idle sessions and caps sessions around 12h. Colab Pro's background
execution is what actually lets this run unattended.

Checkpoints land in `runs/<experiment>/` on your Drive every `SAVE_STEP` steps, so
an interruption only costs you the steps since the last save.

In [ ]:
%cd /content/research
!python train.py \
    --config_path "{DATASET}/train_config.json" \
    --output_path "{RUNS}"

## 6. If you got disconnected — resume

Run cells 1, 2 and 3 again (new VM, so Drive must be remounted and the code
re-cloned), then run the two cells below.

### 6a. Restore the prepared dataset from Drive

In [ ]:
import subprocess
assert DS_ARCHIVE.is_file(), f'no dataset archive at {DS_ARCHIVE}; redo section 4'
subprocess.run(['tar','xzf',str(DS_ARCHIVE),'-C','/content'], check=True)
print('restored ->', DATASET)
print('config:', (DATASET / 'train_config.json').is_file())

### 6b. Resume from the newest checkpoint

`--restore_path` starts a *new* run folder but carries over the model, optimiser,
step and epoch — so checkpoints accumulate across folders. This searches all of
them for the newest one.

In [ ]:
import glob, os
ckpts = glob.glob(str(RUNS / '*' / '*.pth.tar'))
assert ckpts, f'no checkpoints found under {RUNS}'
latest = max(ckpts, key=os.path.getmtime)
print('resuming from', latest)

%cd /content/research
!python train.py \
    --config_path "{DATASET}/train_config.json" \
    --output_path "{RUNS}" \
    --restore_path "{latest}"

## 7. Export the finished model

Collects the config, vocab and best checkpoint into one `bundle/` folder on Drive
that the pip package can load.

In [ ]:
import glob, os
run_dir = max((d for d in glob.glob(str(RUNS / '*')) if os.path.isdir(d)),
              key=os.path.getmtime)
print('exporting from', run_dir)

%cd /content/research
!python export_bundle.py \
    --run-dir "{run_dir}" \
    --config "{DATASET}/train_config.json" \
    --out "{DRIVE}/bundle"

### Test the bundle

Needs `multi_accent_frontend-0.1.0-py3-none-any.whl` uploaded to your Drive folder.
The accent is `base` because the master lexicon's phones are accent-neutral
keysymbols — see the note at the end of RUNBOOK.md about the realised accents.

In [ ]:
# The wheel can be in the repo or on Drive; use whichever is present.
import glob
_whl = (glob.glob(str(CODE / '**' / '*.whl'), recursive=True)
        or glob.glob(str(DRIVE / '*.whl')))
assert _whl, f'no .whl found in {CODE} or {DRIVE} -- commit it to the repo to run this cell'
print('installing', _whl[0])
!pip -q install "{_whl[0]}"
from multi_accent_frontend import Frontend
fe = Frontend.from_pretrained(str(DRIVE / 'bundle'))
for a in ACCENTS:
    print(f"{a}: {fe.transcribe('the bath water ran for hours', accent=a)}")